In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
class PositionalEncoding(nn.Module):
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        super().__init__()

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        self.num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(self.num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(self.num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [3]:
class CrossMultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, encoder_output: torch.Tensor, decoder_input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if encoder_output.shape[-1] != self.d_model:
            raise ValueError(f"Expected encoder input feature dimension to be d_model ({self.d_model}), but got {encoder_output.shape[-1]}")

        if decoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected decoder input feature dimension to be d_model ({self.d_model}), but got {decoder_input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = decoder_input.shape[0]
        encoder_num_tokens = encoder_output.shape[1]
        decoder_num_tokens = decoder_input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(decoder_input)
        k = self.w_k(encoder_output)
        v = self.w_v(encoder_output)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, decoder_num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()

        # Recombining all the heads
        sim3 = sim3.view(batch_size, decoder_num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [4]:
class MaskedMultiHeadAttention(nn.Module):
    """
    Compute Masked Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:

        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Creating a matrix where lower triangle is 1, rest is 0
        mask = torch.tril(torch.ones(num_tokens, num_tokens))
        # Dimension: (num_queries, num_keys)

        # Applying the mask to our attention scores
        sim1 = sim1.masked_fill(mask == 0, float('-inf'))

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [5]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [6]:
class DecoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim 
        self.cross_attention_layer = CrossMultiHeadAttention(self.d_model, self.num_heads)
        self.masked_attention_layer = MaskedMultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim, dropout_rate=dropout_rate)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.norm3 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.masked_attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----CROSS ATTENTION----
        cross_attn_input = self.norm2(x)
        # Normalizing

        cross_attn_output = self.cross_attention_layer(encoder_output, cross_attn_input)
        # Passing though cross attention layer

        x = x + self.dropout(cross_attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm3(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [7]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
hidden_dim = 16

In [8]:
dummy_block_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_block_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_block_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.5642, 0.8576, 0.0809, 0.7148, 0.4712, 0.8294, 0.4697, 0.0178],
         [0.6968, 0.9178, 0.0912, 0.3776, 0.3287, 0.7986, 0.4903, 0.4553],
         [0.7393, 0.7661, 0.8240, 0.8455, 0.0300, 0.0442, 0.2669, 0.0668]],

        [[0.9971, 0.0952, 0.5868, 0.1859, 0.0642, 0.6988, 0.2691, 0.5556],
         [0.2123, 0.9761, 0.4069, 0.8802, 0.5026, 0.8697, 0.9121, 0.2015],
         [0.0604, 0.7031, 0.4763, 0.3540, 0.2233, 0.8634, 0.6757, 0.8982]]])

In [9]:
decoder = DecoderBlock(d_model, num_heads, hidden_dim)
decoder

DecoderBlock(
  (cross_attention_layer): CrossMultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (masked_attention_layer): MaskedMultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm3): LayerNorm((8,), e

In [10]:
dummy_output = decoder(dummy_block_input, dummy_block_input)
dummy_output

tensor([[[ 0.2345,  1.2305,  0.5706,  1.1490,  0.7383,  0.8214,  0.1419,
          -0.5188],
         [ 0.2058,  0.9819,  0.5674,  1.0720,  0.6387,  0.7781,  0.4149,
          -0.3722],
         [ 0.6873,  1.4717,  0.9960,  1.4641,  0.4627,  0.1651,  0.3991,
          -0.8528]],

        [[ 0.7798,  0.0035,  0.9391,  0.6015,  0.4525,  0.4060,  0.4299,
           0.0334],
         [ 0.2673,  1.0846,  0.6897,  1.0613,  0.7208,  1.0355,  0.5185,
          -0.8289],
         [-0.2854,  0.6609,  1.1552,  0.2686,  0.2233,  0.4647,  0.4724,
           0.5123]]], grad_fn=<AddBackward0>)

In [11]:
# Now, Stack Multiple Decoders:
class DecoderStack(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, hidden_dim) for _ in range(num_blocks)])
        self.norm_1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)
        
    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        # Dimension of x: (batch_size, num_tokens, d_model)

        for layer in self.layers:
            x = layer(x, encoder_output)
        output = self.norm_1(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        return output

In [12]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 5
hidden_dim = 16
num_blocks = 6

In [22]:
dummy_stack_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_stack_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_stack_input

torch.Size([2, 3, 8])
(batch_size, num_tokens)


tensor([[[0.9578, 0.8010, 0.6258, 0.0037, 0.6833, 0.1967, 0.0483, 0.4980],
         [0.3200, 0.4806, 0.2962, 0.9539, 0.9909, 0.7047, 0.6465, 0.4233],
         [0.7529, 0.9229, 0.6276, 0.1836, 0.3215, 0.7298, 0.9492, 0.1241]],

        [[0.0165, 0.6462, 0.7665, 0.2054, 0.1431, 0.6611, 0.5199, 0.8307],
         [0.7336, 0.0688, 0.1593, 0.4449, 0.4109, 0.5726, 0.4700, 0.2489],
         [0.3256, 0.4731, 0.2248, 0.5730, 0.8269, 0.2230, 0.1654, 0.0289]]])

In [23]:
decoder_stack = DecoderStack(d_model, num_heads, hidden_dim, num_blocks)
decoder_stack

DecoderStack(
  (layers): ModuleList(
    (0-5): 6 x DecoderBlock(
      (cross_attention_layer): CrossMultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (masked_attention_layer): MaskedMultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((8,)

In [24]:
decoder_stack_output = decoder_stack(dummy_stack_input, dummy_stack_input)
print(decoder_stack_output.shape)
decoder_stack_output

torch.Size([2, 3, 8])


tensor([[[ 1.5774,  0.9476, -0.8187, -1.4323,  0.6289, -1.0529, -0.3592,
           0.5091],
         [ 1.7052,  0.4304, -1.0477, -0.8978,  1.2019, -0.7541, -0.9460,
           0.3081],
         [ 1.4248,  1.5564, -0.8201, -1.1970,  0.0539, -0.2483,  0.3505,
          -1.1202]],

        [[-0.0524, -0.3216,  1.0951, -1.0239, -1.9283,  1.0410,  0.8556,
           0.3346],
         [ 1.9634, -0.6700, -0.4750, -1.2472, -0.4541, -0.6157,  0.4281,
           1.0706],
         [ 2.3296, -1.1544, -0.4393, -0.8536,  0.4473, -0.1025, -0.3164,
           0.0892]]], grad_fn=<NativeLayerNormBackward0>)